#### 1579. Remove Max Number of Edges to Keep Graph Fully Traversable
* https://leetcode.com/problems/remove-max-number-of-edges-to-keep-graph-fully-traversable/description/

In [ ]:
from typing import List

class UnionFind:
    """
    Disjoint Set Union with:
    - Path Compression
    - Union by Rank
    - Component tracking
    """
    def __init__(self, n: int):
        self.parent = list(range(n + 1))
        self.rank = [1] * (n + 1)
        self.components = n  # Track number of connected components

    def find(self, node: int) -> int:
        """Find root with path compression."""
        if self.parent[node] != node:
            self.parent[node] = self.find(self.parent[node])
        return self.parent[node]

    def union(self, u: int, v: int) -> bool:
        """
        Union by rank.
        Returns True if union happened (edge used),
        False if already connected (redundant edge).
        """
        root_u = self.find(u)
        root_v = self.find(v)

        if root_u == root_v:
            return False

        # Union by rank
        if self.rank[root_u] < self.rank[root_v]:
            self.parent[root_u] = root_v
        elif self.rank[root_u] > self.rank[root_v]:
            self.parent[root_v] = root_u
        else:
            self.parent[root_v] = root_u
            self.rank[root_u] += 1

        self.components -= 1
        return True


class Solution:
    def maxNumEdgesToRemove(self, n: int, edges: List[List[int]]) -> int:
        """
        Greedy + Union-Find approach.
        We process edges in linear time and use near-constant DSU operations, 
        giving an overall time complexity of O(E α(n)) ≈ O(E) and space complexity of O(n)

        Steps:
        1. Process type 3 edges first (shared edges).
        2. Then process type 1 (Alice) and type 2 (Bob).
        3. Count edges actually used.
        4. Validate connectivity.
        """

        # Create two DSUs for Alice and Bob
        alice_uf = UnionFind(n)
        bob_uf = UnionFind(n)

        edges_used = 0

        # Step 1: Use type 3 edges first
        for edge_type, u, v in edges:
            if edge_type == 3:
                used_by_alice = alice_uf.union(u, v)
                used_by_bob = bob_uf.union(u, v)

                if used_by_alice or used_by_bob:
                    edges_used += 1

        # Step 2: Use type 1 edges (Alice only)
        for edge_type, u, v in edges:
            if edge_type == 1:
                if alice_uf.union(u, v):
                    edges_used += 1

        # Step 3: Use type 2 edges (Bob only)
        for edge_type, u, v in edges:
            if edge_type == 2:
                if bob_uf.union(u, v):
                    edges_used += 1

        # Step 4: Check if both graphs are fully connected
        if alice_uf.components != 1 or bob_uf.components != 1:
            return -1

        # Removable edges = total - used
        return len(edges) - edges_used